# Project 08: HMM - The Viterbi Algorithm

## Provided Data Structures

In [18]:
import numpy as np
class HMM:
    """Class for HMM

        Attributes:
        init_probs (dictionary): initial probabilities (probability of starting in each state) 
        trans_probs (dictionary): transition probabilities (probability of moving from one state to another)
        emit_probs (dictionary): emission probabilities (probability of observing a symbol in a given state)
        states (list): list of states

        probability_matrix (numpy matrix): observations x state (max probability of that observation given previous and state)
        traceback_matrix (numpy matrix): observations x state (where the max probability for that observation came from)
    """
    def __init__(self, init_probs, trans_probs, emit_probs):
        """Initialize the object.
        """
        self.init = init_probs
        self.trans = trans_probs
        self.emit = emit_probs
        self.states = list(emit_probs.keys())

    def viterbi(self, sequence):
        """Runs Viterbi algorithm for given sequence

        Args:
            sequence (string): DNA sequence to analyze

        Returns:
        probability_matrix (numpy matrix): observations x state (max probability of that observation given previous and state)
        traceback_matrix (numpy matrix): observations x state (where the max probability for that observation came from)
        """

        # create dictionary that stores nucleotide characters by their position in the seqeuce (indexed by 1)
        sequence_position_dict = {}
        for position in range(len(sequence)):
            sequence_position_dict[position] = sequence[position]

        # create traceback matrix with 0's
        traceback_matrix = np.zeros([len(self.states), len(sequence)], dtype=int)

        # create probability matrix of 0's with correct size
        probability_matrix =  np.zeros((len(self.states), len(sequence)))

        # initialize first column with initial probabilities
        for row_num in range(len(self.states)):
            state = self.states[row_num]
            probability_matrix[row_num, 0] = np.log(self.init[state] * self.emit[state][sequence[0]])

        # iterate through remaining columns, finding max probability at each row position
        for column_num in (range(1, len(sequence))):
            current_nucletotide = sequence_position_dict[column_num]
            for row_num in range(len(self.states)):
                current_state = self.states[row_num]
                last_column = column_num - 1
                log_prob_dictionary = {}
                for last_row_num in range(len(self.states)):
                    last_state = self.states[last_row_num]
                    log_last_state_prob = probability_matrix[last_row_num, last_column]
                    log_prob =  log_last_state_prob + np.log(self.trans[last_state][current_state] * self.emit[current_state][current_nucletotide])
                    log_prob_dictionary[last_state] = log_prob
                max_state = max(log_prob_dictionary, key=log_prob_dictionary.get)
                probability_matrix[row_num, column_num] = log_prob_dictionary[max_state]
                traceback_matrix[row_num, column_num] = self.states.index(max_state)

                                
        # Termination
        best_last_row = np.argmax(probability_matrix[:, -1])
        best_prob = probability_matrix[best_last_row, -1]

        # Traceback
        path_rows = [best_last_row]
        current_row = best_last_row
        for column_num in range(len(sequence) - 1, 0, -1):   # last column down to 1
            previous_row = traceback_matrix[current_row, column_num]
            path_rows.append(previous_row)
            current_row = previous_row

        path_rows.reverse()
        path = [self.states[i] for i in path_rows]

        return path, best_prob, probability_matrix, traceback_matrix 
    
    


In [ ]:


# Example observation sequence
obs = "GGCACTGAA"
obs2 = "ATCGATCGATCGATCGATTTATTGCGCATGCATGCTGCGCGCGCGCGCATTTTTACTGCGAT"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

my_hmm = HMM(init_probs, trans_probs, emit_probs)


path, best_prob, probability_matrix, traceback_matrix = my_hmm.viterbi(obs2)

print(path)

